In [1]:
from pathlib import Path
import os
import io

import numpy as np
import pandas as pd
from PIL import Image

from tqdm.auto import tqdm

# If you already used this project-root finder, you can reuse it; otherwise:
def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise RuntimeError(f"Could not find project root starting from {start}")

cwd = Path.cwd()
PROJECT_ROOT = find_project_root(cwd)

META_DIR = PROJECT_ROOT / "data" / "processed" / "metadata"

# A4 input CSV (realistic Emuru)
A4_CSV_PATH = META_DIR / "domain_classification_sentences_A4_emuru_realistic.csv"

# A5 output CSV
A5_CSV_PATH = META_DIR / "domain_classification_sentences_A5_emuru_realistic_noisy.csv"

print("CWD         :", cwd)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("A4 CSV path :", A4_CSV_PATH)
print("A5 CSV path :", A5_CSV_PATH)

df_A4 = pd.read_csv(A4_CSV_PATH)
print("\nA4 shape:", df_A4.shape)
print("A4 columns:", list(df_A4.columns))
print("hf_split values:", df_A4["hf_split"].unique())
print("source values:", df_A4["source"].unique())
print("domain_label counts:\n", df_A4["domain_label"].value_counts())


CWD         : /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/notebooks
PROJECT_ROOT: /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project
A4 CSV path : /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/data/processed/metadata/domain_classification_sentences_A4_emuru_realistic.csv
A5 CSV path : /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/data/processed/metadata/domain_classification_sentences_A5_emuru_realistic_noisy.csv

A4 shape: (19602, 7)
A4 columns: ['filepath', 'source', 'label', 'text', 'idx', 'hf_split', 'domain_label']
hf_split values: ['test' 'train' 'validation']
source values: ['iam' 'emuru']
domain_label counts:
 domain_label
0    9801
1    9801
Name: count, dtype: int64


/home/woody/iwi5/iwi5384h/software/private/conda/envs/emuru/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Where to store the new A5 noisy Emuru images (relative to PROJECT_ROOT)
A5_EMURU_DIR = PROJECT_ROOT / "data" / "raw" / "emuru_A5_noisy"
A5_EMURU_DIR.mkdir(parents=True, exist_ok=True)
print("A5 Emuru image output dir:", A5_EMURU_DIR)

def load_image(path: Path) -> Image.Image:
    img = Image.open(path)
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img

def add_gaussian_noise(img: Image.Image, sigma: float = 8.0) -> Image.Image:
    """
    Add Gaussian noise (std = sigma, in [0,255] space) to an RGB image.
    """
    arr = np.array(img).astype("float32")
    noise = np.random.normal(loc=0.0, scale=sigma, size=arr.shape).astype("float32")
    arr_noisy = arr + noise
    arr_noisy = np.clip(arr_noisy, 0, 255).astype("uint8")
    return Image.fromarray(arr_noisy, mode="RGB")

def slight_blur(img: Image.Image, radius: float = 0.8) -> Image.Image:
    from PIL import ImageFilter
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def jpeg_compress_decompress(img: Image.Image, quality: int = 70) -> Image.Image:
    """
    Simulate JPEG compression artefacts by going through an in-memory
    JPEG encode/decode cycle.
    """
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def noisy_realistic_pipeline(img: Image.Image) -> Image.Image:
    """
    A5 noise pipeline:
    1) small blur
    2) Gaussian noise
    3) light JPEG recompression

    The parameters are intentionally mild: we want images still readable,
    but with extra 'scan-like' noise/degradation vs A4.
    """
    # 1) slight blur
    out = slight_blur(img, radius=0.8)

    # 2) Gaussian noise (std about 8 / 255 ≈ 0.03)
    out = add_gaussian_noise(out, sigma=8.0)

    # 3) JPEG artefacts
    out = jpeg_compress_decompress(out, quality=75)

    return out


A5 Emuru image output dir: /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/data/raw/emuru_A5_noisy


In [3]:
rows_A5 = []

n_emuru = 0
n_emuru_processed = 0

for i, row in tqdm(df_A4.iterrows(), total=len(df_A4)):
    row = row.copy()

    src = row["source"]
    fp_rel = Path(row["filepath"])
    img_in_path = PROJECT_ROOT / fp_rel

    if src == "emuru" and int(row["domain_label"]) == 1:
        n_emuru += 1

        # load original realistic Emuru image
        img = load_image(img_in_path)

        # apply A5 noise pipeline
        img_noisy = noisy_realistic_pipeline(img)

        # build new relative path under A5_EMURU_DIR
        # e.g. if original is data/raw/emuru/emu_00001.png
        # we might save as data/raw/emuru_A5_noisy/emu_00001_noisy.png
        orig_name = fp_rel.name
        stem, ext = os.path.splitext(orig_name)
        new_name = f"{stem}_A5noisy{ext if ext else '.png'}"

        new_rel_path = Path("data") / "raw" / "emuru_A5_noisy" / new_name
        img_out_path = PROJECT_ROOT / new_rel_path
        img_out_path.parent.mkdir(parents=True, exist_ok=True)

        # save noisy image
        img_noisy.save(img_out_path)

        # update filepath for A5 CSV
        row["filepath"] = str(new_rel_path.as_posix())

        n_emuru_processed += 1

    # append row (IAM rows unchanged, Emuru rows point to new noisy image)
    rows_A5.append(row)

df_A5 = pd.DataFrame(rows_A5)

print("A5 shape:", df_A5.shape)
print("domain_label counts:\n", df_A5["domain_label"].value_counts())
print("source counts:\n", df_A5["source"].value_counts())
print("Emuru rows detected:", n_emuru)
print("Emuru rows processed (noisy):", n_emuru_processed)

df_A5.head()


100%|██████████| 19602/19602 [06:53<00:00, 47.35it/s]


A5 shape: (19602, 7)
domain_label counts:
 domain_label
0    9801
1    9801
Name: count, dtype: int64
source counts:
 source
iam      9801
emuru    9801
Name: count, dtype: int64
Emuru rows detected: 9801
Emuru rows processed (noisy): 9801


,filepath,source,label,text,idx,hf_split,domain_label
0,data/raw/iam/iam_07957.png,iam,genuine,commerce may be kept going - though if ever,7957,test,0
1,data/raw/iam/iam_09341.png,iam,genuine,"' Be silent , woman , and listen , ' Band Appa...",9341,test,0
2,data/raw/iam/iam_03294.png,iam,genuine,we would be altogether clearer in our minds,3294,train,0
3,data/raw/iam/iam_03657.png,iam,genuine,enough tacks and he got only the middle hammer...,3657,train,0
4,data/raw/emuru_A5_noisy/iam_04302_sentence_A5n...,emuru,fake,"purpose , the sides from one , and the bottom ...",4302,train,1


In [4]:
df_A5.to_csv(A5_CSV_PATH, index=False)
print("Saved A5 CSV to:", A5_CSV_PATH)


Saved A5 CSV to: /home/hpc/iwi5/iwi5384h/projects/handwriting_forge_project/data/processed/metadata/domain_classification_sentences_A5_emuru_realistic_noisy.csv
